In [ ]:
DRUG = "AZM"
MODEL_CODE = "dfdf"

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.preprocessing import OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
import pandas as pd
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

dataset = pd.read_csv(f"dataset/by_antibiotic_12comorbdims/{DRUG}")

# pca = make_pipeline(
#     PCA(n_components=10)
# )

encoder = OrdinalEncoder()
X = dataset.drop(columns=["anon_id","order_time_jittered_utc_shifted","resistant","Unnamed: 0"])
X = encoder.fit_transform(X)
X = np.nan_to_num(X, nan=-99999)
# X = pca.fit_transform(X)

y = dataset["resistant"].to_numpy()


X = scaler.fit_transform(X, y)
# X, y = resample(X, y, n_samples=50000, replace=False, stratify=y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [ ]:
from qiskit_machine_learning.algorithms.classifiers import VQC

vqc = VQC.from_dill(f"QML Models/{MODEL_CODE}.model")

In [ ]:
import shap

background_profile = np.mean(X_train, axis=0).reshape(1, -1)
exp = shap.PermutationExplainer(vqc.predict_proba, background_profile)

In [ ]:
shapley = exp(X_test)
with open("Shapley/{DRUG}.txt", "w+") as file:
    file.write(str(shapley))
# shap.plots.beeswarm(shapley, max_display=20)